# Standard Benchmark Suite (Lite) — AutoFillGraph v7

External benchmark harness for **AutoFillGraph v7** (Prototype7 backbone).

**Benchmarks:**
- `FUNSD` — official scanned-form dataset (~10 MB, 200 docs)
- `XFUND-DE` — multilingual form benchmark, single-language sample (<=60 docs)

**Scored metrics:**
- `mapping_acc` — form label -> AutoFillGraph property mapping accuracy
- `fill_acc` — exact-fill accuracy after learning from benchmark Q/A pairs
- `abstain_acc` — correct UNKNOWN for unsupported fields

**LLM baseline:** Mistral-small given identical form context in-prompt (no KG, no memory).

Dataset cap: **2 GB** total across all downloaded benchmark data.

In [1]:
from __future__ import annotations
import json, os, re, zipfile
from collections import OrderedDict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    plt = None
    HAS_MPL = False

ROOT = Path.cwd().parent if Path.cwd().name == 'Baseline' else Path.cwd()
CORE_NOTEBOOK = ROOT / 'Baseline' / 'Prototype7.ipynb'
BENCH_ROOT = ROOT / 'data' / 'standard_benchmarks_lite'
BENCH_ROOT.mkdir(parents=True, exist_ok=True)

MAX_DATASET_BYTES = 2 * 1024**3

RUN_FUNSD = True
RUN_XFUND = True
XFUND_LANG = 'de'
XFUND_MAX_DOCS = {'train': 40, 'val': 20}
LLM_BASELINE_MAX_DOCS = 15   # FUNSD docs for Mistral baseline

BASE_EMBEDDER_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MULTILINGUAL_EMBEDDER_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

FUNSD_URL = 'https://guillaumejaume.github.io/FUNSD/dataset.zip'
XFUND_BASE_URL = 'https://github.com/doc-analysis/XFUND/releases/download/v1.0/'

# Expanded patterns covering personal-profile fields + generic FUNSD/XFUND labels
QUESTION_PATTERNS = OrderedDict([
    ('full_name',       [r'\bfull\s*name\b', r'\bname\b', r'applicant', r'customer', r'employee']),
    ('email',           [r'e-?mail', r'mail address']),
    ('work_email',      [r'institutional e-?mail', r'work e-?mail']),
    ('phone',           [r'phone', r'mobile', r'telephone', r'tel\.?', r'cell']),
    ('address',         [r'address', r'street', r'mailing']),
    ('city',            [r'\bcity\b', r'town']),
    ('state',           [r'\bstate\b', r'province', r'region']),
    ('zip_code',        [r'zip', r'postal']),
    ('country',         [r'country of residence', r'country']),
    ('citizenship',     [r'citizenship', r'nationality']),
    ('passport_number', [r'passport']),
    ('visa_status',     [r'\bvisa\b', r'immigration status']),
    ('university',      [r'university', r'college', r'school', r'institution']),
    ('degree',          [r'\bdegree\b', r'\bprogram\b', r'\bmajor\b']),
    ('department',      [r'department', r'faculty']),
    ('gpa',             [r'\bgpa\b', r'\bgrade\b', r'academic score']),
    ('advisor',         [r'advisor', r'supervisor', r'mentor']),
    ('graduation_date', [r'graduation', r'date of completion', r'completion date']),
    ('employer',        [r'employer', r'company', r'organization']),
    ('job_title',       [r'job title', r'\bposition\b', r'\brole\b']),
    ('bank_name',       [r'bank name', r'\bbank\b']),
    ('tax_id',          [r'tax id', r'\btin\b', r'\bssn\b']),
    ('signature',       [r'signature', r'sign here', r'signed by']),
    ('date',            [r'\bdate\b', r'dated', r'as of']),
    ('amount',          [r'\bamount\b', r'total', r'\bsum\b']),
    ('reference',       [r'reference', r'\bref\b', r'order number']),
    ('description',     [r'description', r'\bdetails\b', r'\bitem\b']),
    ('quantity',        [r'\bqty\b', r'quantity', r'\bunits\b']),
])

CANONICAL_PROMPTS = {
    'full_name': 'Full Name', 'email': 'Email', 'work_email': 'Work Email',
    'phone': 'Phone Number', 'address': 'Current Address', 'city': 'City',
    'state': 'State', 'zip_code': 'ZIP Code', 'country': 'Country',
    'citizenship': 'Citizenship', 'passport_number': 'Passport Number',
    'visa_status': 'Visa Status', 'university': 'University',
    'degree': 'Degree Program', 'department': 'Academic Department',
    'gpa': 'GPA', 'advisor': 'Advisor', 'graduation_date': 'Graduation Date',
    'employer': 'Current Employer', 'job_title': 'Job Title',
    'bank_name': 'Bank Name', 'tax_id': 'Tax ID',
    'signature': 'Signature', 'date': 'Date', 'amount': 'Total Amount',
    'reference': 'Reference Number', 'description': 'Description',
    'quantity': 'Quantity',
}


def norm_text(v: str) -> str:
    t = re.sub(r'[^a-z0-9@.+/\-]+', ' ', str(v or '').lower())
    return re.sub(r'\s+', ' ', t).strip()


def human_bytes(n: int) -> str:
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    s = float(n)
    for u in units:
        if s < 1024 or u == units[-1]:
            return f'{s:.1f} {u}'
        s /= 1024.0


def dir_size_bytes(p: Path) -> int:
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) if p.exists() else 0


def assert_under_budget(root: Path = BENCH_ROOT, cap: int = MAX_DATASET_BYTES) -> int:
    total = dir_size_bytes(root)
    if total > cap:
        raise RuntimeError(f'Budget exceeded: {human_bytes(total)} > {human_bytes(cap)}')
    return total


def exactish_match(pred, gold) -> bool:
    p, g = norm_text(pred), norm_text(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    return re.sub(r'[^a-z0-9]+', '', p) == re.sub(r'[^a-z0-9]+', '', g)


def infer_expected_prop(question: str) -> Optional[str]:
    q = norm_text(question)
    for prop, patterns in QUESTION_PATTERNS.items():
        if any(re.search(pat, q) for pat in patterns):
            return prop
    return None


print(f'Core notebook : {CORE_NOTEBOOK}  exists={CORE_NOTEBOOK.exists()}')
print(f'Benchmark root: {BENCH_ROOT}')
print(f'Budget cap    : {human_bytes(MAX_DATASET_BYTES)}')
print(f'Current usage : {human_bytes(assert_under_budget())}')


C:\Users\govin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Core notebook : E:\Autofill-Graph\Baseline\Prototype7.ipynb  exists=True
Benchmark root: E:\Autofill-Graph\data\standard_benchmarks_lite
Budget cap    : 2.0 GB
Current usage : 0.0 B


In [2]:
def _filter_cell(cell_id: str, src: str) -> str:
    # Strip eager init / API probes / module-level instantiation from Prototype7
    lines = src.splitlines()

    if cell_id == 'cell-init':
        skip_tokens = [
            'Loading MiniLM', 'SentenceTransformer(', 'MiniLM ready',
            'Tesseract', 'CLIP importable', '_t0 = time.perf_counter',
        ]
        return '\n'.join(l for l in lines if not any(tok in l for tok in skip_tokens))

    if cell_id == 'cell-components':
        filtered, in_probe = [], False
        for line in lines:
            if line.strip().startswith('if LLM.available()'):
                in_probe = True
                continue
            if in_probe:
                if 'All components defined' in line:
                    in_probe = False
                    filtered.append(line)
                continue
            filtered.append(line)
        return '\n'.join(filtered)

    if cell_id == 'cell-agent':
        marker = '\nAGENT = AutoFillAgent('
        return src.split(marker, 1)[0].rstrip() if marker in src else src

    return src


def load_prototype7_core(nb_path: Path) -> None:
    payload = json.loads(nb_path.read_text(encoding='utf-8'))
    cells_src = {c.get('id'): ''.join(c.get('source', [])) for c in payload['cells']}
    for cid in ['cell-init', 'cell-components', 'cell-agent']:
        if cid not in cells_src:
            raise KeyError(f'Missing cell id {cid!r} in {nb_path.name}')
        exec(_filter_cell(cid, cells_src[cid]), globals())  # noqa: S102


load_prototype7_core(CORE_NOTEBOOK)

# Per-benchmark caches: avoid re-encoding 46 properties for every document
EMBEDDER_CACHE: dict = {}
MAPPER_CACHE: dict = {}


def get_embedder(model_name: str):
    if model_name not in EMBEDDER_CACHE:
        print(f'  Loading embedder: {model_name}')
        EMBEDDER_CACHE[model_name] = SentenceTransformer(model_name)
    return EMBEDDER_CACHE[model_name]


def get_mapper(model_name: str):
    if model_name not in MAPPER_CACHE:
        print(f'  Building FieldMapper: {model_name}')
        MAPPER_CACHE[model_name] = FieldMapper(get_embedder(model_name))
    return MAPPER_CACHE[model_name]


def build_fresh_agent(embedder_model: str, use_real_llm: bool = False):
    # Fresh KG+memory; shares cached embedder and mapper (no property re-encoding per doc)
    emb = get_embedder(embedder_model)
    llm_key = API_KEY if use_real_llm else ''
    llm_obj = MistralClient(llm_key, DEFAULT_MODEL)
    agent = AutoFillAgent(emb, llm_obj)
    agent.mapper = get_mapper(embedder_model)
    return agent


print('\n\u2705 Prototype7 core loaded (no eager model init, no API probe).')
print(f'   API_KEY present: {bool(API_KEY)} | model: {DEFAULT_MODEL}')


✅ NetworkX 3.4.2
✅ Schema: 43 properties, 8 layers
✅ All components defined

✅ Prototype7 core loaded (no eager model init, no API probe).
   API_KEY present: True | model: mistral-small-latest


In [3]:
def download_file(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f'  Cached: {dest.name}')
        return dest
    print(f'  Downloading: {url}')
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with dest.open('wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return dest


def unzip_file(zip_path: Path, out_dir: Path) -> Path:
    if out_dir.exists() and any(out_dir.iterdir()):
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    with __import__('zipfile').ZipFile(zip_path) as zf:
        zf.extractall(out_dir)
    return out_dir


def prepare_funsd(root: Path = BENCH_ROOT) -> Path:
    ds_dir = root / 'funsd'
    download_file(FUNSD_URL, ds_dir / 'dataset.zip')
    unzip_file(ds_dir / 'dataset.zip', ds_dir / 'raw')
    print(f'  FUNSD ready | usage: {human_bytes(assert_under_budget())}')
    return ds_dir


def prune_xfund(json_path: Path, image_root: Path, max_docs: int) -> None:
    data = json.loads(json_path.read_text(encoding='utf-8'))
    docs = data.get('documents', [])
    if len(docs) <= max_docs:
        return
    keep = docs[:max_docs]
    keep_rel = {Path(d['img']['fname']).as_posix() for d in keep}
    data['documents'] = keep
    json_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
    for p in list(image_root.rglob('*')):
        if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            if p.relative_to(image_root).as_posix() not in keep_rel:
                p.unlink()


def prepare_xfund(lang: str = XFUND_LANG, max_docs: Optional[dict] = None,
                  root: Path = BENCH_ROOT) -> dict:
    max_docs = max_docs or XFUND_MAX_DOCS
    ds_dir = root / 'xfund' / lang
    ds_dir.mkdir(parents=True, exist_ok=True)
    out = {}
    for split in ['train', 'val']:
        jp = ds_dir / f'{lang}.{split}.json'
        zp = ds_dir / f'{lang}.{split}.zip'
        idir = ds_dir / f'{split}_images'
        download_file(f'{XFUND_BASE_URL}{lang}.{split}.json', jp)
        download_file(f'{XFUND_BASE_URL}{lang}.{split}.zip', zp)
        unzip_file(zp, idir)
        if max_docs.get(split):
            prune_xfund(jp, idir, max_docs[split])
        out[split] = {'json': jp, 'images': idir}
        print(f'  XFUND-{lang.upper()} {split} ready | usage: {human_bytes(assert_under_budget())}')
    return out


def dataset_inventory(root: Path = BENCH_ROOT) -> pd.DataFrame:
    rows = [{'dataset_dir': p.name, 'size_bytes': dir_size_bytes(p),
              'size_human': human_bytes(dir_size_bytes(p))}
             for p in sorted(root.iterdir())] if root.exists() else []
    return pd.DataFrame(rows)


In [4]:
def extract_linked_pairs(items: List[dict]) -> List[dict]:
    by_id = {item['id']: item for item in items if 'id' in item}
    links: set = set()
    for item in items:
        for link in item.get('linking', []):
            if isinstance(link, list) and len(link) == 2:
                links.add(tuple(sorted(link)))
    pairs, seen = [], set()
    for a, b in sorted(links):
        if a not in by_id or b not in by_id:
            continue
        ia, ib = by_id[a], by_id[b]
        la = str(ia.get('label', '')).lower()
        lb = str(ib.get('label', '')).lower()
        if {la, lb} != {'question', 'answer'}:
            continue
        question = ia.get('text', '') if la == 'question' else ib.get('text', '')
        answer   = ib.get('text', '') if la == 'question' else ia.get('text', '')
        question, answer = str(question).strip(), str(answer).strip()
        if not question or not answer:
            continue
        key = (norm_text(question), norm_text(answer))
        if key in seen:
            continue
        seen.add(key)
        pairs.append({'question': question, 'answer': answer})
    return pairs


def parse_funsd_annotation(path: Path) -> List[dict]:
    payload = json.loads(path.read_text(encoding='utf-8'))
    return extract_linked_pairs(payload.get('form', []))


def iter_funsd_documents(funsd_dir: Path):
    raw_root = funsd_dir / 'raw' / 'dataset'
    for split, sub in [('train', 'training_data'), ('test', 'testing_data')]:
        ann_dir = raw_root / sub / 'annotations'
        for ap in sorted(ann_dir.glob('*.json')):
            yield {'dataset': 'FUNSD', 'split': split,
                   'doc_id': ap.stem, 'pairs': parse_funsd_annotation(ap)}


def parse_xfund_json(json_path: Path) -> List[dict]:
    payload = json.loads(json_path.read_text(encoding='utf-8'))
    out = []
    for doc in payload.get('documents', []):
        pairs = extract_linked_pairs(doc.get('document', []))
        doc_id = doc.get('id') or Path(doc.get('img', {}).get('fname', 'unk')).stem
        out.append({'doc_id': str(doc_id), 'pairs': pairs})
    return out


def iter_xfund_documents(prepared: dict, lang: str):
    for split, parts in prepared.items():
        for doc in parse_xfund_json(parts['json']):
            yield {'dataset': f'XFUND-{lang.upper()}', 'split': split,
                   'doc_id': doc['doc_id'], 'pairs': doc['pairs']}


In [5]:
def evaluate_document(doc: dict, embedder_model: str, max_unsup: int = 3):
    # Full AutoFillGraph v7 pipeline evaluation on one form document
    agent = build_fresh_agent(embedder_model, use_real_llm=False)
    supported = OrderedDict()
    unsupported: List[str] = []
    mapping_rows: List[dict] = []

    for pair in doc['pairs']:
        q, gold = pair['question'], pair['answer']
        exp_prop = infer_expected_prop(q)
        if exp_prop:
            pred_prop, phase, score = agent.mapper.map(q)  # 3-tuple in Prototype7
            mapping_rows.append({
                'dataset': doc['dataset'], 'split': doc['split'], 'doc_id': doc['doc_id'],
                'question': q, 'expected_prop': exp_prop, 'predicted_prop': pred_prop,
                'phase': phase, 'score': round(score, 3),
                'ok': pred_prop == exp_prop,
            })
            if exp_prop not in supported:
                supported[exp_prop] = (q, gold)
        else:
            unsupported.append(q)

    if supported:
        agent.learn({raw_q: ans for raw_q, ans in supported.values()}, context='benchmark')

    fill_rows: List[dict] = []
    for prop, (raw_q, gold) in supported.items():
        query = CANONICAL_PROMPTS.get(prop, prop.replace('_', ' ').title())
        ep = agent.autofill([query], domain='general', use_llm=False)
        res = ep.results[query]
        is_filled = res.status != FillStatus.UNKNOWN and str(res.value).strip()
        ok = bool(is_filled) and exactish_match(res.value, gold)
        fill_rows.append({
            'dataset': doc['dataset'], 'split': doc['split'], 'doc_id': doc['doc_id'],
            'raw_question': raw_q, 'query': query,
            'expected_prop': prop, 'predicted_prop': res.prop,
            'expected_value': gold, 'predicted_value': res.value,
            'status': res.status.value if hasattr(res.status, 'value') else str(res.status),
            'route': res.route.value if hasattr(res.route, 'value') else str(res.route),
            'ok': ok,
        })

    abstain_rows: List[dict] = []
    for q in unsupported[:max_unsup]:
        ep = agent.autofill([q], domain='general', use_llm=False)
        res = ep.results[q]
        is_unk = (res.status == FillStatus.UNKNOWN
                  or str(res.value).strip().upper() == 'UNKNOWN')
        abstain_rows.append({
            'dataset': doc['dataset'], 'split': doc['split'], 'doc_id': doc['doc_id'],
            'question': q, 'predicted_prop': res.prop, 'predicted_value': res.value,
            'status': res.status.value if hasattr(res.status, 'value') else str(res.status),
            'ok': is_unk,
        })

    return mapping_rows, fill_rows, abstain_rows


def run_benchmark(doc_iter, benchmark_name: str, embedder_model: str,
                  max_docs: Optional[int] = None) -> dict:
    mapping_rows, fill_rows, abstain_rows = [], [], []
    docs_seen = docs_with_eval = 0
    for doc in doc_iter:
        if max_docs is not None and docs_seen >= max_docs:
            break
        docs_seen += 1
        m, f, a = evaluate_document(doc, embedder_model)
        if f or a:
            docs_with_eval += 1
        mapping_rows.extend(m)
        fill_rows.extend(f)
        abstain_rows.extend(a)
        if docs_seen % 25 == 0 or docs_seen <= 2:
            print(f'  [{benchmark_name}] {docs_seen} docs | '
                  f'mapping={len(mapping_rows)} fill={len(fill_rows)} abstain={len(abstain_rows)}')

    mdf = pd.DataFrame(mapping_rows)
    fdf = pd.DataFrame(fill_rows)
    adf = pd.DataFrame(abstain_rows)

    summary = {
        'benchmark': benchmark_name,
        'embedder': embedder_model.split('/')[-1],
        'docs_seen': docs_seen,
        'docs_with_eval': docs_with_eval,
        'mapping_n': len(mdf),
        'mapping_acc': float(mdf['ok'].mean()) if len(mdf) else float('nan'),
        'fill_n': len(fdf),
        'fill_acc': float(fdf['ok'].mean()) if len(fdf) else float('nan'),
        'abstain_n': len(adf),
        'abstain_acc': float(adf['ok'].mean()) if len(adf) else float('nan'),
    }
    return {'summary': summary, 'mapping': mdf, 'fill': fdf, 'abstain': adf}


In [6]:
prepared = {}

if RUN_FUNSD:
    print('Preparing FUNSD...')
    prepared['FUNSD'] = prepare_funsd()

if RUN_XFUND:
    print(f'\nPreparing XFUND-{XFUND_LANG.upper()}...')
    prepared[f'XFUND-{XFUND_LANG.upper()}'] = prepare_xfund(XFUND_LANG, XFUND_MAX_DOCS)

inv = dataset_inventory()
display(inv if len(inv) else pd.DataFrame(columns=['dataset_dir', 'size_bytes', 'size_human']))
print(f'\nTotal local benchmark usage: {human_bytes(assert_under_budget())}')


Preparing FUNSD...
  Downloading: https://guillaumejaume.github.io/FUNSD/dataset.zip


  FUNSD ready | usage: 42.6 MB

Preparing XFUND-DE...
  Downloading: https://github.com/doc-analysis/XFUND/releases/download/v1.0/de.train.json


  Downloading: https://github.com/doc-analysis/XFUND/releases/download/v1.0/de.train.zip


  XFUND-DE train ready | usage: 288.2 MB
  Downloading: https://github.com/doc-analysis/XFUND/releases/download/v1.0/de.val.json


  Downloading: https://github.com/doc-analysis/XFUND/releases/download/v1.0/de.val.zip


  XFUND-DE val ready | usage: 383.8 MB


,dataset_dir,size_bytes,size_human
0,funsd,44718596,42.6 MB
1,xfund,357701502,341.1 MB



Total local benchmark usage: 383.8 MB


In [7]:
results = {}

if RUN_FUNSD:
    print('\n=== FUNSD Benchmark ===')
    results['FUNSD'] = run_benchmark(
        iter_funsd_documents(prepared['FUNSD']),
        benchmark_name='FUNSD',
        embedder_model=BASE_EMBEDDER_MODEL,
    )

if RUN_XFUND:
    xname = f'XFUND-{XFUND_LANG.upper()}'
    print(f'\n=== {xname} Benchmark ===')
    results[xname] = run_benchmark(
        iter_xfund_documents(prepared[xname], XFUND_LANG),
        benchmark_name=xname,
        embedder_model=MULTILINGUAL_EMBEDDER_MODEL,
    )

summary_df = pd.DataFrame([p['summary'] for p in results.values()])
print('\n=== Benchmark Summary ===')
display(summary_df)

for name, pack in results.items():
    s = pack['summary']
    print(f'\n{name}: mapping={s["mapping_acc"]:.3f} (n={s["mapping_n"]}) | '
          f'fill={s["fill_acc"]:.3f} (n={s["fill_n"]}) | '
          f'abstain={s["abstain_acc"]:.3f} (n={s["abstain_n"]})')
    if len(pack['fill']):
        print('\nSample fill results:')
        display(pack['fill'][['doc_id','query','expected_value','predicted_value','route','ok']].head(8))



=== FUNSD Benchmark ===
  Loading embedder: sentence-transformers/all-MiniLM-L6-v2


  Building FieldMapper: sentence-transformers/all-MiniLM-L6-v2


  [FUNSD] 1 docs | mapping=3 fill=3 abstain=3


  [FUNSD] 2 docs | mapping=11 fill=9 abstain=6


  [FUNSD] 25 docs | mapping=101 fill=57 abstain=75


...

,benchmark,embedder,docs_seen,docs_with_eval,mapping_n,mapping_acc,fill_n,fill_acc,abstain_n,abstain_acc
0,FUNSD,all-MiniLM-L6-v2,199,194,783,0.425287,407,0.538084,573,0.984293
1,XFUND-DE,paraphrase-multilingual-MiniLM-L12-v2,60,60,303,0.547855,123,0.837398,180,0.872222



FUNSD: mapping=0.425 (n=783) | fill=0.538 (n=407) | abstain=0.984 (n=573)

Sample fill results:


,doc_id,query,expected_value,predicted_value,route,ok
0,0000971160,Date,9/ 3/ 92,9/ 3/ 92,local,True
1,0000971160,Full Name,"M. Hamann P. Harper, P. Martinez",9/ 3/ 92,local,False
2,0000971160,Advisor,J. S. Wigand,J. S. Wigand,local,True
3,0000989556,Current Employer,B. A. T. CYPRUS,B. A. T. CYPRUS,local,True
4,0000989556,Country,CYPRUS,CYPRUS,local,True
5,0000989556,Date,"May 1, 1985","May 1, 1985",local,True
6,0000989556,Total Amount,90 mm,UNKNOWN,local,False
7,0000989556,Job Title,27 mm,27 mm,local,True



XFUND-DE: mapping=0.548 (n=303) | fill=0.837 (n=123) | abstain=0.872 (n=180)

Sample fill results:


,doc_id,query,expected_value,predicted_value,route,ok
0,de_train_0,Full Name,Julian Herrmann,Julian Herrmann,local,True
1,de_train_0,Phone Number,16724891042,16724891042,local,True
2,de_train_1,Phone Number,Max Weber,UNKNOWN,local,False
3,de_train_1,Full Name,"""Max Weber, 55232 Alzey, Bahnhofstraße 30","""Max Weber, 55232 Alzey, Bahnhofstraße 30",local,True
4,de_train_2,Phone Number,Erstbestellung (mit Bild).,UNKNOWN,local,False
5,de_train_2,Email,Jonas321@gmail.com,Jonas321@gmail.com,local,True
6,de_train_3,Full Name,Max Weber,Max Weber,local,True
7,de_train_3,Phone Number,17830921231,17830921231,local,True


In [8]:
# LLM-only baseline: Mistral-small receives the same Q/A pairs AutoFillGraph learns from.
# Both systems get identical information; the difference is memory architecture.
def llm_baseline_funsd(funsd_dir: Path, max_docs: int = LLM_BASELINE_MAX_DOCS) -> pd.DataFrame:
    llm = MistralClient(API_KEY, DEFAULT_MODEL)
    if not llm.available():
        print('WARNING: No API key found. LLM baseline skipped.')
        return pd.DataFrame()

    rows: List[dict] = []
    docs_done = 0
    for doc in iter_funsd_documents(funsd_dir):
        if docs_done >= max_docs:
            break
        matched = [(p, infer_expected_prop(p['question'])) for p in doc['pairs']]
        matched = [(p, prop) for p, prop in matched if prop]
        if not matched:
            continue
        docs_done += 1

        context = '\n'.join(
            f'  "{p["question"]}": {p["answer"]}' for p, _ in matched
        )

        for pair, prop in matched:
            canonical = CANONICAL_PROMPTS.get(prop, pair['question'])
            prompt = (
                'A filled form contains the following fields:\n'
                f'{context}\n\n'
                f'What is the value for the field "{canonical}"?\n'
                'Reply with only the exact value string, nothing else.'
            )
            predicted = llm._post(
                [{'role': 'user', 'content': prompt}],
                json_mode=False,
                timeout=40,
            ).strip()
            rows.append({
                'doc_id': doc['doc_id'],
                'split': doc['split'],
                'question': pair['question'],
                'canonical': canonical,
                'expected_prop': prop,
                'expected_value': pair['answer'],
                'llm_predicted': predicted,
                'ok': exactish_match(predicted, pair['answer']),
            })
        print(f'  LLM baseline: {docs_done}/{max_docs} docs | pairs so far: {len(rows)}')

    df = pd.DataFrame(rows)
    if len(df):
        acc = df['ok'].mean()
        print(f'\nMistral-small fill accuracy: {acc:.3f}  ({df["ok"].sum()}/{len(df)} pairs, {docs_done} FUNSD docs)')
        display(df[['doc_id', 'canonical', 'expected_value', 'llm_predicted', 'ok']].head(10))
    return df


print(f'Running LLM baseline on {LLM_BASELINE_MAX_DOCS} FUNSD docs (Mistral-small, in-context) ...')
llm_baseline_df = llm_baseline_funsd(prepared['FUNSD'])


Running LLM baseline on 15 FUNSD docs (Mistral-small, in-context) ...


  LLM baseline: 1/15 docs | pairs so far: 3


  LLM baseline: 2/15 docs | pairs so far: 11


  LLM baseline: 3/15 docs | pairs so far: 14


  LLM baseline: 4/15 docs | pairs so far: 23


...

  LLM baseline: 12/15 docs | pairs so far: 65


  LLM baseline: 13/15 docs | pairs so far: 69


  LLM baseline: 14/15 docs | pairs so far: 72


  LLM baseline: 15/15 docs | pairs so far: 76

Mistral-small fill accuracy: 0.434  (33/76 pairs, 15 FUNSD docs)


,doc_id,canonical,expected_value,llm_predicted,ok
0,0000971160,Date,9/ 3/ 92,9/3/92,True
1,0000971160,Full Name,"M. Hamann P. Harper, P. Martinez","M. Hamann P. Harper, P. Martinez",True
2,0000971160,Advisor,J. S. Wigand,J. S. Wigand,True
3,0000989556,Current Employer,B. A. T. CYPRUS,B. A. T. CYPRUS,True
4,0000989556,Country,CYPRUS,CYPRUS,True
5,0000989556,Date,"May 1, 1985","May 1, 1985",True
6,0000989556,Total Amount,90 mm,110 mm,False
7,0000989556,Total Amount,110 mm,974 mg,False
8,0000989556,Job Title,27 mm,B. A. T. CYPRUS,False
9,0000989556,Total Amount,974 mg,974 mg,True


In [9]:
import math

afg_fill = results.get('FUNSD', {}).get('summary', {}).get('fill_acc', float('nan'))
afg_map  = results.get('FUNSD', {}).get('summary', {}).get('mapping_acc', float('nan'))
afg_abs  = results.get('FUNSD', {}).get('summary', {}).get('abstain_acc', float('nan'))
llm_fill = float(llm_baseline_df['ok'].mean()) if len(llm_baseline_df) else float('nan')

comp = pd.DataFrame([
    {
        'System': 'AutoFillGraph v7 (KG + Adaptive Routing)',
        'FUNSD Fill Acc': round(afg_fill, 3) if not math.isnan(afg_fill) else float('nan'),
        'FUNSD Mapping Acc': round(afg_map, 3) if not math.isnan(afg_map) else float('nan'),
        'Abstain Acc': round(afg_abs, 3) if not math.isnan(afg_abs) else float('nan'),
        'LLM API Calls (fill)': 0,
    },
    {
        'System': 'Mistral-small (in-context, no memory)',
        'FUNSD Fill Acc': round(llm_fill, 3) if not math.isnan(llm_fill) else float('nan'),
        'FUNSD Mapping Acc': float('nan'),
        'Abstain Acc': float('nan'),
        'LLM API Calls (fill)': int(llm_baseline_df['ok'].count()) if len(llm_baseline_df) else 0,
    },
])
print('\n=== AutoFillGraph v7 vs LLM Baseline (FUNSD) ===')
display(comp)

if not math.isnan(afg_fill) and not math.isnan(llm_fill):
    lift = afg_fill - llm_fill
    print(f'\nFill accuracy lift (AutoFillGraph - Mistral): {lift:+.3f}')
print('AutoFillGraph uses 0 LLM API calls for fill; Mistral baseline uses 1 call per field.')



=== AutoFillGraph v7 vs LLM Baseline (FUNSD) ===


,System,FUNSD Fill Acc,FUNSD Mapping Acc,Abstain Acc,LLM API Calls (fill)
0,AutoFillGraph v7 (KG + Adaptive Routing),0.538,0.425,0.984,0
1,"Mistral-small (in-context, no memory)",0.434,NaN,NaN,76



Fill accuracy lift (AutoFillGraph - Mistral): +0.104
AutoFillGraph uses 0 LLM API calls for fill; Mistral baseline uses 1 call per field.


In [10]:
import math, numpy as np

if results and HAS_MPL:
    plot_df = pd.DataFrame([p['summary'] for p in results.values()])
    metrics = ['mapping_acc', 'fill_acc', 'abstain_acc']
    labels  = ['Mapping Acc', 'Fill Acc', 'Abstain Acc']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Left: per-benchmark grouped bar
    x = np.arange(len(plot_df))
    w = 0.22
    colors = ['#4C72B0', '#DD8452', '#55A868']
    for i, (m, lab) in enumerate(zip(metrics, labels)):
        vals = plot_df[m].fillna(0.0).to_numpy()
        ax1.bar(x + i * w, vals, w, label=lab, color=colors[i])
    ax1.set_xticks(x + w)
    ax1.set_xticklabels(plot_df['benchmark'].tolist(), fontsize=9)
    ax1.set_ylim(0, 1.15)
    ax1.set_ylabel('Accuracy')
    ax1.set_title('AutoFillGraph v7 - External Benchmarks')
    ax1.legend(fontsize=8)
    ax1.axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.5)

    # Right: FUNSD fill acc comparison
    systems = ['AutoFillGraph v7\n(KG+Routing)', 'Mistral-small\n(in-context)']
    vals2   = [afg_fill if not math.isnan(afg_fill) else 0,
               llm_fill if not math.isnan(llm_fill) else 0]
    bars = ax2.bar(systems, vals2, color=['#4C72B0', '#C44E52'], width=0.4)
    for bar, val in zip(bars, vals2):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax2.set_ylim(0, 1.15)
    ax2.set_ylabel('Fill Accuracy (FUNSD)')
    ax2.set_title('Fill Accuracy: AutoFillGraph v7 vs LLM Baseline')
    ax2.axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.5)

    plt.tight_layout()
    out_fig = BENCH_ROOT / 'benchmark_summary.png'
    fig.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {out_fig}')

print('\n=== Metric Definitions ===')
print('mapping_acc : form label correctly mapped to AutoFillGraph v7 property')
print('fill_acc    : Q/A pair value correctly retrieved after learning (local KG, 0 API calls)')
print('abstain_acc : out-of-schema field correctly returned as UNKNOWN')

print('\n=== Caveats for Paper ===')
print('- FUNSD/XFUND are scanned business forms; AutoFillGraph targets personal-profile autofill.')
print('  Cross-domain transfer partially limits fill_acc.')
print('- LLM baseline needs 1 API call per field; AutoFillGraph uses 0 (local KG retrieval).')
print('- XFUND limited to <=60 German docs to stay within 2 GB data budget.')
print('- Both systems receive identical Q/A information; difference is memory architecture.')

# Save CSV artefacts
for name, pack in results.items():
    safe = name.replace('-', '_').lower()
    for key in ['fill', 'mapping', 'abstain']:
        pack[key].to_csv(BENCH_ROOT / f'{safe}_{key}.csv', index=False)
if len(llm_baseline_df):
    llm_baseline_df.to_csv(BENCH_ROOT / 'llm_baseline_funsd.csv', index=False)
print(f'\nCSV results saved to {BENCH_ROOT}')


Figure saved: E:\Autofill-Graph\data\standard_benchmarks_lite\benchmark_summary.png

=== Metric Definitions ===
mapping_acc : form label correctly mapped to AutoFillGraph v7 property
fill_acc    : Q/A pair value correctly retrieved after learning (local KG, 0 API calls)
abstain_acc : out-of-schema field correctly returned as UNKNOWN

=== Caveats for Paper ===
- FUNSD/XFUND are scanned business forms; AutoFillGraph targets personal-profile autofill.
  Cross-domain transfer partially limits fill_acc.
- LLM baseline needs 1 API call per field; AutoFillGraph uses 0 (local KG retrieval).
- XFUND limited to <=60 German docs to stay within 2 GB data budget.
- Both systems receive identical Q/A information; difference is memory architecture.

CSV results saved to E:\Autofill-Graph\data\standard_benchmarks_lite
